# Import and optimization

In [1]:
import pandas as pd
import numpy as np
import math

import seaborn as sns
import missingno
import matplotlib.pyplot as plt
from statsmodels.stats.outliers_influence import variance_inflation_factor
import plotly.express as px
import phik
from pygeodesy import GeoidKarney, LatLon_

from sklearn.model_selection import train_test_split, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

# CatBoost
from catboost import CatBoostRegressor, Pool

# Опционально: оптимизация гиперпараметров
import optuna

# Визуализация
from plotly.offline import init_notebook_mode, plot
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.interpolate import griddata

import warnings
warnings.filterwarnings("ignore")

In [2]:
class CFG:
    TARGET = ['GGFM_f']  # Предполагаем, что это вектор (phi, dh)
    N_FOLDS = 5
    RANDOM_STATE = 3

    COLORADO_PATH = './Data/Synt/Colorado_60000_merge.csv'
    NSO_PATH = './Data/Synt/NSO_60000_merge.csv'

In [3]:
class DataLoader:
    def __init__(self, colorado: pd.DataFrame, nso: pd.DataFrame):
        self.colorado = colorado
        self.nso = nso
        self.log_features = []
        self.X = None
        self.y = None
    
    @staticmethod
    def reduce_mem_usage(dataframe):
        start_mem = dataframe.memory_usage().sum() / 1024**2
        print(f"Изначальное использование памяти: {start_mem:.2f} MB")
        
        for col in dataframe.columns:
            col_type = dataframe[col].dtype
            
            if str(col_type).startswith('datetime') or str(col_type) == 'category':
                continue
            
            if col_type != object:
                c_min = dataframe[col].min()
                c_max = dataframe[col].max()
                
                if str(col_type)[:3] == 'int':
                    if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                        dataframe[col] = dataframe[col].astype(np.int8)
                    elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                        dataframe[col] = dataframe[col].astype(np.int16)
                    elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                        dataframe[col] = dataframe[col].astype(np.int32)
                    elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                        dataframe[col] = dataframe[col].astype(np.int64)
                else:
                    if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                        dataframe[col] = dataframe[col].astype(np.float16)
                    elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                        dataframe[col] = dataframe[col].astype(np.float32)
                    else:
                        dataframe[col] = dataframe[col].astype(np.float64)
            else:
                dataframe[col] = dataframe[col].astype('category')
        
        end_mem = dataframe.memory_usage().sum() / 1024**2
        print(f"Итоговое использование памяти: {end_mem:.2f} MB")
        print(f"Экономия {(start_mem - end_mem) / start_mem * 100:.1f}%")
        
        return dataframe

    def load(self, option='colorado'):
        print(f'Loading data')
        print(f'Choosed option: {option}')

        if option == 'compare':
            self.colorado = self.colorado.merge(self.nso, left_on='latitude(deg)', right_on='lat(deg)', how='outer')
        elif option == 'colorado':
            pass

        self.colorado = self.reduce_mem_usage(self.colorado)

In [4]:
loader = DataLoader(
    colorado=pd.read_csv(CFG.COLORADO_PATH, skipinitialspace=True, index_col='index'),
    nso=pd.read_csv(CFG.NSO_PATH, skipinitialspace=True, index_col='index')
)

In [5]:
loader.load()

Loading data
Choosed option: colorado
Изначальное использование памяти: 47.24 MB
Итоговое использование памяти: 17.26 MB
Экономия 63.5%


In [6]:
loader.colorado.drop(columns=['GGFM_g'], inplace=True)
loader.colorado.drop_duplicates(inplace=True)
loader.colorado.reset_index(drop=True, inplace=True)

In [8]:
# Предположим, что GGFM_f — это составной столбец или нужно разбить на два: phi и dh
# В оригинале, вероятно, GGFM_f содержит [phi, dh]
# Если это один столбец — нужно уточнить структуру. Пока предположим, что это два столбца:


features = loader.colorado.drop(columns=CFG.TARGET)
target = loader.colorado[CFG.TARGET]

print(f"Features shape: {features.shape}")
print(f"Target shape: {target.shape}")

Features shape: (476233, 10)
Target shape: (476233, 1)


In [10]:
# Стандартизация
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = scaler_X.fit_transform(features)
y_scaled = scaler_y.fit_transform(target)

# Разделение
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_scaled, test_size=0.2, random_state=CFG.RANDOM_STATE
)

# CatBoost Training

In [11]:
# Создаём пулы для CatBoost
train_pool_pot = Pool(X_train, y_train)
test_pool_pot = Pool(X_test, y_test)

In [12]:
# Функция для подбора гиперпараметров (опционально)
def objective_potential(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'random_seed': CFG.RANDOM_STATE,
        'verbose': False,
        'loss_function': 'RMSE',
        'task_type': 'GPU',
    }
    
    model = CatBoostRegressor(**params)
    model.fit(train_pool_pot)
    pred = model.predict(test_pool_pot)
    return mean_squared_error(y_test, pred)


# Подбор (раскомментируйте, если нужно)
study_pot = optuna.create_study(direction='minimize')
study_pot.optimize(objective_potential, n_trials=20)

print(f"Best params for potential: {study_pot.best_params}")

[I 2025-08-28 19:52:17,191] A new study created in memory with name: no-name-ff66f57b-a16a-4f65-8cbf-b6ad873fdc8a
[I 2025-08-28 19:52:19,644] Trial 0 finished with value: 9.832073199966873e-05 and parameters: {'iterations': 391, 'learning_rate': 0.026699614144994145, 'depth': 6, 'l2_leaf_reg': 4.016198533823406}. Best is trial 0 with value: 9.832073199966873e-05.
[I 2025-08-28 19:52:22,416] Trial 1 finished with value: 6.105972271914329e-05 and parameters: {'iterations': 508, 'learning_rate': 0.11927946897327643, 'depth': 7, 'l2_leaf_reg': 6.549904655137552}. Best is trial 1 with value: 6.105972271914329e-05.
[I 2025-08-28 19:52:30,916] Trial 2 finished with value: 3.2416982001247316e-05 and parameters: {'iterations': 940, 'learning_rate': 0.22403605716040537, 'depth': 10, 'l2_leaf_reg': 6.559422120750609}. Best is trial 2 with value: 3.2416982001247316e-05.
[I 2025-08-28 19:52:36,182] Trial 3 finished with value: 3.8590917245831326e-05 and parameters: {'iterations': 717, 'learning_rat

Best params for potential: {'iterations': 968, 'learning_rate': 0.29796415116814656, 'depth': 10, 'l2_leaf_reg': 7.804396993514237}


In [13]:
print("Обучение модели для потенциала...")
model_pot = CatBoostRegressor(
    iterations=500,
    learning_rate=0.1,
    depth=6,
    l2_leaf_reg=3,
    random_seed=CFG.RANDOM_STATE,
    verbose=100,
    loss_function='RMSE'
)
model_pot.fit(train_pool_pot, eval_set=test_pool_pot)

Обучение модели для потенциала...
0:	learn: 0.9036596	test: 0.9060392	best: 0.9060392 (0)	total: 23.5ms	remaining: 11.7s
100:	learn: 0.0154904	test: 0.0155055	best: 0.0155055 (100)	total: 1.44s	remaining: 5.68s
200:	learn: 0.0112593	test: 0.0112691	best: 0.0112691 (200)	total: 2.8s	remaining: 4.17s
300:	learn: 0.0091441	test: 0.0091682	best: 0.0091682 (300)	total: 4.14s	remaining: 2.74s
400:	learn: 0.0078353	test: 0.0078642	best: 0.0078642 (400)	total: 5.49s	remaining: 1.36s
499:	learn: 0.0069895	test: 0.0070194	best: 0.0070194 (499)	total: 6.86s	remaining: 0us

bestTest = 0.007019404005
bestIteration = 499



In [14]:
# Предсказания
phi_pred = model_pot.predict(X_test)

# Денормализация
phi_pred_2d = np.column_stack((phi_pred, np.zeros_like(phi_pred)))

phi_original = scaler_y.inverse_transform(phi_pred_2d)

# Тестовые значения
phi_test_original = scaler_y.inverse_transform(
    np.column_stack((y_test, np.zeros_like(y_test)))
)

# Разности
phi_diff = phi_original - phi_test_original

print("Пример предсказаний:")
print("Потенциал (φ):", phi_original[:5])

Пример предсказаний:
Потенциал (φ): [[62365920.23200744 62378779.64176359]
 [62373342.097905   62378779.64176359]
 [62361809.34971575 62378779.64176359]
 [62361058.88511415 62378779.64176359]
 [62384311.94642605 62378779.64176359]]


# Визуализация

In [18]:
init_notebook_mode(connected=True)

# Координаты (долгота, широта) — предполагаем, что X_test содержит их в первых двух колонках
longitudes = scaler_X.inverse_transform(X_test)[:, 1]  # долгота
latitudes = scaler_X.inverse_transform(X_test)[:, 0]   # широта

def create_surface(x, y, z, title, colorscale='Viridis'):
    # Принудительно делаем 1D
    x = np.array(x).flatten()
    y = np.array(y).flatten()
    z = np.array(z).flatten()
    
    # Проверяем NaN
    mask = ~np.isnan(x) & ~np.isnan(y) & ~np.isnan(z)
    x, y, z = x[mask], y[mask], z[mask]
    
    # Создаём сетку
    grid_x, grid_y = np.mgrid[
        np.min(x):np.max(x):100j,
        np.min(y):np.max(y):100j
    ]
    
    try:
        grid_z = griddata((x, y), z, (grid_x, grid_y), method='cubic', fill_value=np.nanmean(z))
    except Exception as e:
        print("Cubic interpolation failed:", e)
        grid_z = griddata((x, y), z, (grid_x, grid_y), method='linear', fill_value=np.nanmean(z))
    
    return go.Surface(
        x=grid_x, y=grid_y, z=grid_z,
        colorscale=colorscale, opacity=0.8,
        surfacecolor=grid_z, name=title
    )

fig = make_subplots(
    rows=1, cols=3,
    specs=[[{'type': 'surface'}, {'type': 'surface'}, {'type': 'surface'}]],
    subplot_titles=(
        'Предсказанный потенциал', 'Тестовый потенциал', 'Разность потенциалов',
    ),
    horizontal_spacing=0.05,
    vertical_spacing=0.1
)

fig.add_trace(create_surface(longitudes, latitudes, phi_original, 'Предсказанный потенциал'), row=1, col=1)
fig.add_trace(create_surface(longitudes, latitudes, phi_test_original, 'Тестовый потенциал'), row=1, col=2)
fig.add_trace(create_surface(longitudes, latitudes, phi_diff, 'Разность потенциалов', 'RdBu'), row=1, col=3)

fig.update_layout(
    title_text='3D визуализация гравитационного поля (CatBoost)',
    scene=dict(xaxis_title='Долгота', yaxis_title='Широта', zaxis_title='Потенциал (м²/с²)'),
    scene2=dict(xaxis_title='Долгота', yaxis_title='Широта', zaxis_title='Потенциал (м²/с²)'),
    scene3=dict(xaxis_title='Долгота', yaxis_title='Широта', zaxis_title='Разность (м²/с²)'),
    width=1700, height=1100, margin=dict(r=50, l=50, b=50, t=50)
)

fig.show()
plot(fig, filename='catboost_gravity_comparison.html', auto_open=False)

ValueError: operands could not be broadcast together with shapes (95247,) (190494,) 

In [15]:
pd.DataFrame(phi_diff).describe().T\
    .style.bar(subset=['mean'], color=px.colors.qualitative.G10[1])\
    .background_gradient(subset=['std'])\
    .background_gradient(subset=['50%'])

<style>...</style>

In [16]:
# Сохранение моделей
model_pot.save_model("catboost_potential.cbm")

# Сохранение скалеров (через joblib)
import joblib
joblib.dump(scaler_X, "scaler_X.pkl")
joblib.dump(scaler_y, "scaler_y.pkl")